In [0]:
import os
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.window import Window
BASE_DIR = "/Volumes/data/default/cbs_data"


In [0]:

def load_csvs_to_dict(base_dir):
    """
    Scans a directory for CSV files and loads them into a dictionary of DataFrames.
    
    Args:
        base_dir (str): The DBFS or Volume path to scan.
        
    Returns:
        dict: A dictionary where keys are filenames and values are Spark DataFrames.
    """
    df_dict = {}
    
    print(f"--- Accessing Path: {base_dir} ---")
    
    try:
        # List files using Databricks utilities
        files = dbutils.fs.ls(base_dir)
        csv_files = [f for f in files if f.name.endswith(".csv")]
        
        if not csv_files:
            print("Warning: No CSV files found in the specified directory.")
            return df_dict

        for f in csv_files:
            # Create a clean key (remove .csv extension)
            clean_name = f.name.replace(".csv", "")
            
            try:
                # Load the DataFrame
                df = (spark.read
                      .format("csv")
                      .option("header", "true")
                      .option("inferSchema", "true")
                      .load(f.path))
                
                df_dict[clean_name] = df
                print(f"✅ Loaded: '{clean_name}' [Rows: {df.count()}]")
                
            except Exception as e:
                print(f"❌ Failed to load '{f.name}': {e}")

    except Exception as e:
        print(f"Critical Error accessing directory: {e}")

    return df_dict
def Master_Data(BASE_DIR):
    dataframes = load_csvs_to_dict(BASE_DIR)
    if dataframes:
        dataframes["transactions"] = dataframes["transactions"].join(dataframes["accounts"], on="account_id", how="inner")
        dataframes["transactions"] = dataframes["transactions"].join(dataframes["customers"], on="customer_id", how="inner")
        Master_File = dataframes["transactions"]
        print(f"--- Master File: ---")
        print (f"✅ File Contains {Master_File.count()} rows")
    return Master_File




In [0]:
#(Task 1 A)

def get_customer_total_balance(BASE_DIR):
    ''' The aim of this function is to determine the total balance of each customer.
    The function should return a Spark DataFrame with three columns: customer_id, account type and total_balance.
    The total_balance column should contain the sum of all account balances for each customer.
    '''
    Master_File = Master_Data(BASE_DIR)
    Master_File = Master_File.groupBy("customer_id", "account_type").agg({"balance": "sum"}).withColumnRenamed("sum(balance)", "total_balance")
    Master_File.select(col("customer_id"), col("account_type"), col("total_balance")).display()
    
    #display(Master_File)
    return Master_File

get_customer_total_balance(BASE_DIR)  
    


In [0]:

#TASK 1B
def new_account_last_year():
    ''' The aim of this function is to determine the number of new accounts opened in the last year.
    The function should return a Spark DataFrame with five column: customer_id, first_name, Last_name,account_id and opening_date. The Max_Date points at the year 2024 therefore the opening date should capture new account within this range 2023-09-17
    '''
    # Load the data (assuming Master_Data returns a DataFrame)
    Master_File = Master_Data(BASE_DIR)
    # 1. Get the Max Date (2024-09-17)
    # Using F.max is cleaner than the agg dict syntax
    Max_date = Master_File.select(F.max("opening_date")).collect()[0][0]
    if Max_date is None:
        print("No data found in Master_File.")
        return None

    # 2. Calculate the Start Date (one year prior)
    # add_months handles leap years and different month lengths better than manual math
    start_date = F.add_months(F.lit(Max_date), -12)
    
    # 3. Filter and Select specific columns
    # We filter where opening_date is between 2023-09-17 and 2024-09-17
    result_df = Master_File.filter(
        (F.col("opening_date") > start_date) & 
        (F.col("opening_date") <= F.lit(Max_date))
    ).select(
        "customer_id", 
        "first_name", 
        "last_name", 
        "account_id", 
        "opening_date"
    )
    
    # Optional: Count and show for debugging
    print(f"Max Date: {Max_date}")
    print(f"Filtering for accounts after: 2023-09-17")
    print(f"Total new accounts: {result_df.count()}")
    result_df = result_df.dropDuplicates(["account_id"])
    return result_df

# To run it and see the result in Databricks:
df_new_accounts = new_account_last_year()
display(df_new_accounts)

In [0]:
#TASK 1C
def TopN_customer_total_balance(BASE_DIR):
    ''' Get TopN customer with highest total balance across all accounts
    The function should return a Spark DataFrame with three columns: First_Name, Last_Name and total_balance.
    The total_balance column should contain the sum of all account balances for each customer
    '''

    Master_File = Master_Data(BASE_DIR)
    topn = Master_File.groupBy( "first_name", "last_name").agg({"balance": "sum"}).withColumnRenamed("sum(balance)", "total_balance").orderBy(F.desc("total_balance")).limit(5)
    topn.select(col( "first_name"), col( "last_name"), col( "total_balance")).display()
    return topn

BASE_DIR = "/Volumes/data/default/cbs_data"
TopN_customer_total_balance(BASE_DIR)


In [0]:
#TASK 2A
def greater_than_500():
    ''' The aim of this function is to determine the number of transactions with an amount greater than 500 within the last 30 days.
    The function will return a Spark DataFrame with six columns: account_id, customer_id, first_name, last_name, amount and transaction_date.
    '''
    Master_File = Master_Data(BASE_DIR)
    Max_date = Master_File.select(F.max("transaction_date")).collect()[0][0]
    if Max_date is None:
        print("No data found in Master_File.")
        return None
    '''
    2. Calculate the Start Date (one year prior)
    A) date_add is being utilise to subtract 30 days from the Max-date to filter for the last 30 days
    B) Filter dataset by transaction_date (filters out the largest chunk of data)
    C) transaction_type has been filter for rows that are equal to "Withdrawal" - This reduces amout of rows futher
    D) Amount has been filter for rows that are greater than $500 (working on a small dataset)
    
    '''
    
    Past_30_days= F.date_add(F.lit(Max_date), -30)
    result_df = Master_File.filter(
        (F.col("transaction_date") > Past_30_days) & 
        (F.col("transaction_date") <= F.lit(Max_date))& 
        (F.col("transaction_type") == "Withdrawal")&
        (F.col("amount") > 500)
        
    ).select(
        "account_id", 
        "customer_id", 
        "first_name", 
        "last_name", 
        "amount", 
        "transaction_date"
    )
    
    # Dataframe description - checking.
    print(f"Max Date: {Max_date}")
    print(f"Filtering for transactions greater than $500 after: 2024-08-24")
    print(f"Total withdrawal amount: {result_df.count()}")
    
    return result_df


df_new_accounts = greater_than_500()
display(df_new_accounts)


In [0]:

#TASK 2B
def Deposit_Analysis_LBH():
    """
    Calculates the total deposit amount per customer within the last 6 months 
    based on the most recent transaction date in the dataset.
    """
    
    # 1. Load Data
    master_file = Master_Data(BASE_DIR)
    
    # 2. Get the reference date (Max date in the dataset)
  
    max_date_row = master_file.select(F.max("transaction_date")).first()
    
    if not max_date_row or max_date_row[0] is None:
        print("No data found in Master_File.")
        return None
    
    max_date = max_date_row[0]
    print(max_date)
    
    # 3. Calculate the Start Date (one year prior)
    start_date = F.add_months(F.lit(max_date), -6)
 
    # 4. Filter, Group, and Aggregate
    # filter for Deposit, Group by customer_id, Sum the amount
    result_df = (
        master_file
        .filter(
            (F.col("transaction_date") >= start_date) &
            (F.col("transaction_date") <= F.lit(max_date))&
            (F.col("transaction_type") == "Deposit")
        )
        .groupBy("customer_id")
        .agg(F.count("transaction_type").alias("total_deposit"))
        #.orderBy(F.desc("total_deposit"))
    )
    
    return result_df
result_df = Deposit_Analysis_LBH()
display(result_df)

In [0]:
#TASK 2C 

def calculate_running_balance():
    # 2. Load Data
    # Ensure transactions.csv is in the working directory
    master_file = Master_Data(BASE_DIR)
    
    # 3. Define the Window
    # We partition by account_id so each account has its own "bucket"
    # We order by date and transaction_id to ensure a consistent timeline
    window_spec = Window.partitionBy("account_id").orderBy("transaction_date", "transaction_id")
    
    # 4. Calculate Running Balance
    # Logic: Cumulative sum of 'amount' from the start of time to the current row
    df_with_running = master_file.withColumn(
        "running_balance", 
        F.round(F.sum("amount").over(window_spec), 2)
    )
    
    # 5. Calculate Prior Balance and Reconstruction for verification
    # prior_balance is the running balance minus the current transaction amount
    result_df = df_with_running.withColumn(
        "prior_balance",
        F.round(F.col("running_balance") - F.col("amount"), 2)
    ).withColumn(
        "reconstructed_delta",
        F.round(F.col("running_balance") - F.col("prior_balance"), 2)
    )
    result_df = result_df.select(
    "account_id", 
    "transaction_id",
    "transaction_date",
    "transaction_type", 
    "amount",
    "running_balance"
    #"prior_balance",
    #"reconstructed_delta"
    ).orderBy("account_id", "transaction_date")
    return result_df

# --- Execution ---
result = calculate_running_balance()
display(result)

In [0]:
#TASK 2C
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def Get_running_Balance():
    master_file = Master_Data(BASE_DIR)

    # 3. Define the Window Specification
    # Partition by account_id and order by date and ID
    window_spec = Window.partitionBy("account_id") \
                        .orderBy("transaction_date", "transaction_id") \
                        .rowsBetween(Window.unboundedPreceding, Window.currentRow)

    # 4. Calculate Running Balance
    # This maps the CASE WHEN logic directly to PySpark
    df_with_balance = master_file.withColumn(
        "running_balance",
        F.sum(
            F.when(F.col("transaction_type") == "Deposit", F.col("amount"))
            .when(F.col("transaction_type") == "Withdrawal", -F.col("amount"))
            .when(F.col("transaction_type") == "Payment", F.col("amount"))
            .when(F.col("transaction_type") == "Transfer", F.col("amount"))
            .otherwise(0)
        ).over(window_spec)
    )

    # 5. Final Select and Order
    result = df_with_balance.select(
        "account_id",
        "transaction_id",
        "transaction_date",
        "amount",
        "transaction_type",
        F.round("running_balance", 2).alias("running_balance")
    ).orderBy("account_id", "transaction_date")

    # Show the results
    return result

result = Get_running_Balance()
display(result)

In [0]:
#TASK3A

from pyspark.sql import functions as F

def Mean_Analysis():
    """
    Computes the mean transaction and balance amount for each account
    over the last 12 months.
    """
    # 1. Load Data
    master_file = Master_Data(BASE_DIR)
    
    # 2. Determine the date range
    # Get the latest date in the data as a Python date object
    max_date_row = master_file.select(F.max("transaction_date")).first()
    if not max_date_row[0]:
        return None # Handle empty DataFrame case
    
    max_date = max_date_row[0]
    
    # 3. Filter data to the last 12 months
    # We use add_months on the literal max_date to find our cutoff
    mean_df = master_file.filter(
        F.col("transaction_date") >= F.add_months(F.lit(max_date), -12)
    )
    
    # 4. Group by customer_id and calculate means
    mean_df = (
        mean_df.groupBy("customer_id")
        .agg(
            F.mean("amount").alias("avg_transaction_amount"),
            F.mean("balance").alias("avg_balance")
        )
    )
    
    return mean_df

# Execution
mean_df = Mean_Analysis()
if mean_df:
    display(mean_df)

In [0]:
#TASK-3B
def Customer_with_most_transactions():
    """
    Computes the customer with the most transactions in the last 3 months.
    """
    # 1. Load Data
    master_file = Master_Data(BASE_DIR)
    #cleanse and filter data to the last 3 months
    max_date = master_file.select(F.max("transaction_date")).first()[0]
    start_date = F.add_months(F.lit(max_date), -3)  
    # 2. Group by customer_id and count transactions
    result_df = (
        master_file
        .filter(
        
            (F.col("transaction_date") > start_date) &
            (F.col("transaction_date") <= F.lit(max_date))
        )
        .groupBy("customer_id","first_name","last_name")
        .agg(F.count("transaction_id").alias("number_of_transactions"))
        .orderBy(F.desc("number_of_transactions"))
        .limit(1)
    )
    
    return  result_df
most_transactions = Customer_with_most_transactions()
print(f"Customer with the most transactions in the last 3 months: ")
display(most_transactions)




In [0]:
#Task 4 
from pyspark.sql import functions as F


# 1. Calculate the sum of transactions per account
txn_sums = master_file.groupBy("account_id").agg(
    F.sum(
        F.when(F.col("transaction_type") == "Deposit", F.col("amount"))
        .when(F.col("transaction_type") == "Withdrawal", -F.col("amount"))
        .otherwise(0)
    ).alias("calculated_balance")
)

# 2. Get the reported balance
# We assume each account has one current balance. We'll grab the max/latest.
account_balances = master_file.groupBy("account_id").agg(F.max("balance").alias("reported_balance"))

# 3. Join and find discrepancies
discrepancies = txn_sums.join(account_balances, "account_id", "inner")

# 4. Filter for mismatches
# Using round to handle floating point math issues
result = discrepancies.filter(
    F.round(F.col("reported_balance"), 2) != F.round(F.col("calculated_balance"), 2)
)

display(result)


In [0]:
#Task 4B
from pyspark.sql import functions as F

def Missing_Or_Incomplete_Records(): 
    """ 
    Identifies customers with incomplete or missing information.
    Checks: first_name, last_name, date_of_birth, address, and zip (< 5 digits).
    """
    master_file = Master_Data(BASE_DIR)
    
    # 1. Define standard columns to check
    check_cols = ["first_name", "last_name", "date_of_birth", "address"]
    
    # 2. Create expressions for standard null/empty checks
    missing_logic = [
        F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), F.lit(c)).otherwise(None) 
        for c in check_cols
    ]
    
    # 3. Specific logic for ZIP codes (Length < 5)
    
    zip_logic = F.when(
        F.col("zip").isNull() | 
        (F.trim(F.col("zip").cast("string")) == "") | 
        (F.length(F.col("zip").cast("string")) < 5), 
        F.lit("zip")
    ).otherwise(None)
    
    missing_logic.append(zip_logic)
    
    # 4. Build the result dataframe
    result_df = (
        master_file
        .withColumn("missing_fields", F.concat_ws(", ", *missing_logic))
        # Only keep records where at least one field was flagged
        .filter(F.col("missing_fields") != "")
        .select(
            "customer_id", 
            "first_name", 
            "last_name", 
            "missing_fields"
        )
    )
    result_df = result_df.dropDuplicates(["customer_id"]) # remove duplicate rows
    return result_df

# Run and display
incomplete_customers = Missing_Or_Incomplete_Records()
display(incomplete_customers)

In [0]:
#Task 4C 
def Identify_Duplicate_Accounts():
    """
    Identifies customers who have more than one account of the same type.
    Outputs: customer_id, account_type, number_of_duplicates
    """
    # 1. Load the accounts data
    master_accounts = Master_Data(BASE_DIR) 
    
    # 2. Group by customer and type, then count
   # Group by customer_id and account_type, then count occurrences
    duplicate_accounts = master_accounts.groupBy("customer_id", "account_type") \
    .agg(F.count_distinct("account_id").alias("number_of_duplicates")) \
    .filter(F.col("number_of_duplicates") > 1)

# 5. Show results
   # duplicate_accounts.orderBy(F.col("number_of_duplicates").desc()).show()

    # Show the results
    #duplicates_df.show()
    
    # 3. Sort by customer_id for a clean report
    result_df = duplicate_accounts.select(
        "customer_id", 
        "account_type", 
        "number_of_duplicates"
    ).orderBy("Number_of_duplicates")
    
    return result_df

# Execute and display in Databricks
duplicate_report = Identify_Duplicate_Accounts()
display(duplicate_report)

In [0]:
#Task 4D 
def Invalid_Unrecognised_Transaction_Type(): 
    """ The purpose of the function is to identify customer with invalid or unrecognised transaction type (i.e other than Deposit, Withdrawal, Transfer, Payment) """
    master_file = Master_Data(BASE_DIR) 
    
    # Define the list of valid transaction types
    valid_types = ["Deposit", "Withdrawal", "Payment", "Transfer"]

    # Filter for transactions NOT in the valid types list
    invalid_txns = master_file.filter(~F.col("transaction_type").isin(valid_types))

    # Select the required columns for the output
    output = invalid_txns.select("transaction_id", "account_id", "transaction_type")

    # Show the results
    if output.count() > 0:
        return output
    else:
        print("No invalid transactions found.")
        return 
Df =Invalid_Unrecognised_Transaction_Type()
display(Df)


In [0]:
#TASK 4E 

def Identify_Non_Credit_Accounts_with_Negative_Balance(): 
    """ The purpose of the function is to identify customer with non credit account with negative balance """
    master_file = Master_Data(BASE_DIR) 
    negative_balance_accounts = master_file.filter((F.col("account_type") != "Credit") & (F.col("balance") < 0))
    negative_balance_accounts = negative_balance_accounts.select("customer_id", "account_id", "account_type", "balance")
    negative_balance_accounts = negative_balance_accounts.dropDuplicates(["account_id"])
    return negative_balance_accounts
non_cred = Identify_Non_Credit_Accounts_with_Negative_Balance()
display(non_cred)